# Electrolyte Imbalance Prediction from NHANES Questionnaire Features

## Goal
Identify which NHANES questionnaire variables are most useful for predicting the target variable `electrolyte_imbalance`.

## Important constraints
- We want **quiz-eligible** features, not just maximum predictive performance.
- We exclude:
  - target leakage
  - medication / diagnosis / RXD variables
  - survey design variables
  - lab / exam / technical derived variables
  - features that cannot realistically be asked in an online quiz
- The target `electrolyte_imbalance` is treated as a **broad proxy**, not a clinical diagnosis.

## Outputs
- Univariate ranking of questionnaire features
- Model-based feature importance ranking
- Shortlist of quiz candidates

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, pointbiserialr

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [2]:
DATA_PATH = Path("../data/processed/nhanes_merged_adults_final.csv")
OUTPUT_DIR = Path("../data/processed/model_outputs_electrolyte_imbalance")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "electrolyte_imbalance"
RANDOM_STATE = 42
TEST_SIZE = 0.20

# Falls deine Spalte anders heißt, hier anpassen:
MED_COUNT_CANDIDATES = ["med_count", "medication_count", "n_medications", "rx_count"]

print("DATA_PATH:", DATA_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

DATA_PATH: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\nhanes_merged_adults_final.csv
OUTPUT_DIR: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_electrolyte_imbalance


In [3]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Shape:", df.shape)
print("First 10 columns:")
print(df.columns[:10].tolist())

assert TARGET in df.columns, f"{TARGET} not found in dataframe."

print(df[TARGET].value_counts(dropna=False).sort_index())
print()
print("Target prevalence:")
print(df[TARGET].value_counts(normalize=True, dropna=False).sort_index().round(4))

Shape: (7437, 878)
First 10 columns:
['SEQN', 'age_years', 'income_poverty_ratio', 'mec_exam_weight', 'interview_weight', 'survey_psu', 'survey_stratum', 'gender', 'ethnicity', 'education']
electrolyte_imbalance
0    6858
1     579
Name: count, dtype: int64

Target prevalence:
electrolyte_imbalance
0    0.9221
1    0.0779
Name: proportion, dtype: float64


## FE Alcohol

In [14]:
ALCOHOL_RAW_COLS = [
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol",
    "alq121___past_12_mo_how_often_drink_alcoholic_bev",
    "alq130___avg_#_alcoholic_drinks/day___past_12_mos",
    "alq142___#_days_have_4_or_5_drinks/past_12_mos",
    "alq151___ever_have_4/5_or_more_drinks_every_day?",
    "alq170___past_30_days_#_times_4_5_drinks_on_an_oc",
]

available_alcohol_cols = [c for c in ALCOHOL_RAW_COLS if c in df.columns]
missing_alcohol_cols = [c for c in ALCOHOL_RAW_COLS if c not in df.columns]

print("Available alcohol raw cols:", available_alcohol_cols)
print("Missing alcohol raw cols:", missing_alcohol_cols)

Available alcohol raw cols: ['alq111___ever_had_a_drink_of_any_kind_of_alcohol', 'alq121___past_12_mo_how_often_drink_alcoholic_bev', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq142___#_days_have_4_or_5_drinks/past_12_mos', 'alq151___ever_have_4/5_or_more_drinks_every_day?', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc']
Missing alcohol raw cols: []


In [17]:
import numpy as np
import pandas as pd

def clean_nhanes_numeric(series):
    s = pd.to_numeric(series, errors="coerce")
    
    invalid_values = {
        7, 9,
        77, 99,
        777, 999,
        7777, 9999,
        77777, 99999
    }
    
    s = s.where(~s.isin(invalid_values), np.nan)
    return s

for col in available_alcohol_cols:
    df[col] = clean_nhanes_numeric(df[col])

df[available_alcohol_cols].head()

,alq111___ever_had_a_drink_of_any_kind_of_alcohol,alq121___past_12_mo_how_often_drink_alcoholic_bev,alq130___avg_#_alcoholic_drinks/day___past_12_mos,alq142___#_days_have_4_or_5_drinks/past_12_mos,alq151___ever_have_4/5_or_more_drinks_every_day?,alq170___past_30_days_#_times_4_5_drinks_on_an_oc
0,1.0,10.0,1.0,0.0,2.0,0.0
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,0.0,NaN,NaN,1.0,NaN
4,1.0,0.0,NaN,NaN,2.0,NaN


In [18]:
# Ever drank alcohol
if "alq111___ever_had_a_drink_of_any_kind_of_alcohol" in df.columns:
    df["alcohol_ever_yes"] = np.where(
        df["alq111___ever_had_a_drink_of_any_kind_of_alcohol"].isna(),
        np.nan,
        (df["alq111___ever_had_a_drink_of_any_kind_of_alcohol"] == 1).astype(float)
    )

# Any binge drinking in past 12 months
if "alq142___#_days_have_4_or_5_drinks/past_12_mos" in df.columns:
    df["alcohol_binge_any_12m"] = np.where(
        df["alq142___#_days_have_4_or_5_drinks/past_12_mos"].isna(),
        np.nan,
        (df["alq142___#_days_have_4_or_5_drinks/past_12_mos"] > 0).astype(float)
    )

# Any binge occasion in past 30 days
if "alq170___past_30_days_#_times_4_5_drinks_on_an_oc" in df.columns:
    df["alcohol_binge_any_30d"] = np.where(
        df["alq170___past_30_days_#_times_4_5_drinks_on_an_oc"].isna(),
        np.nan,
        (df["alq170___past_30_days_#_times_4_5_drinks_on_an_oc"] > 0).astype(float)
    )

# Ever 4/5+ drinks every day
if "alq151___ever_have_4/5_or_more_drinks_every_day?" in df.columns:
    df["alcohol_ever_daily_heavy"] = np.where(
        df["alq151___ever_have_4/5_or_more_drinks_every_day?"].isna(),
        np.nan,
        (df["alq151___ever_have_4/5_or_more_drinks_every_day?"] == 1).astype(float)
    )

In [19]:
if "alq130___avg_#_alcoholic_drinks/day___past_12_mos" in df.columns:
    drinks = df["alq130___avg_#_alcoholic_drinks/day___past_12_mos"]

    df["alcohol_drinks_per_day_bin"] = pd.cut(
        drinks,
        bins=[-np.inf, 0, 1, 2, 4, np.inf],
        labels=["0", "up_to_1", "up_to_2", "up_to_4", "more_than_4"]
    )

    df["alcohol_drinks_per_day_ge_2"] = np.where(
        drinks.isna(),
        np.nan,
        (drinks >= 2).astype(float)
    )

    df["alcohol_drinks_per_day_ge_4"] = np.where(
        drinks.isna(),
        np.nan,
        (drinks >= 4).astype(float)
    )

In [20]:
if "alq142___#_days_have_4_or_5_drinks/past_12_mos" in df.columns:
    binge_days = df["alq142___#_days_have_4_or_5_drinks/past_12_mos"]

    df["alcohol_binge_days_12m_bin"] = pd.cut(
        binge_days,
        bins=[-np.inf, 0, 1, 12, 52, np.inf],
        labels=["0", "1", "2_to_12", "13_to_52", "53_plus"]
    )

    df["alcohol_binge_monthly_or_more"] = np.where(
        binge_days.isna(),
        np.nan,
        (binge_days >= 12).astype(float)
    )

In [21]:
risk_cols = [
    c for c in [
        "alcohol_binge_any_12m",
        "alcohol_binge_any_30d",
        "alcohol_ever_daily_heavy",
        "alcohol_drinks_per_day_ge_4",
        "alcohol_binge_monthly_or_more",
    ]
    if c in df.columns
]

if risk_cols:
    risk_matrix = df[risk_cols]
    
    df["alcohol_any_risk_signal"] = np.where(
        risk_matrix.isna().all(axis=1),
        np.nan,
        (risk_matrix.fillna(0).max(axis=1) > 0).astype(float)
    )

print("Risk cols used:", risk_cols)

Risk cols used: ['alcohol_binge_any_12m', 'alcohol_binge_any_30d', 'alcohol_ever_daily_heavy', 'alcohol_drinks_per_day_ge_4', 'alcohol_binge_monthly_or_more']


In [22]:
ALCOHOL_FE_COLS = [
    "alcohol_ever_yes",
    "alcohol_binge_any_12m",
    "alcohol_binge_any_30d",
    "alcohol_ever_daily_heavy",
    "alcohol_drinks_per_day_bin",
    "alcohol_drinks_per_day_ge_2",
    "alcohol_drinks_per_day_ge_4",
    "alcohol_binge_days_12m_bin",
    "alcohol_binge_monthly_or_more",
    "alcohol_any_risk_signal",
]

ALCOHOL_FE_COLS = [c for c in ALCOHOL_FE_COLS if c in df.columns]

pd.DataFrame({
    "feature": ALCOHOL_FE_COLS,
    "dtype": [str(df[c].dtype) for c in ALCOHOL_FE_COLS],
    "missing_pct": [round(df[c].isna().mean() * 100, 2) for c in ALCOHOL_FE_COLS],
    "n_unique": [df[c].nunique(dropna=True) for c in ALCOHOL_FE_COLS],
    "example_values": [list(df[c].dropna().unique()[:8]) for c in ALCOHOL_FE_COLS],
})

,feature,dtype,missing_pct,n_unique,example_values
0,alcohol_ever_yes,float64,12.75,2,"[1.0, 0.0]"
1,alcohol_binge_any_12m,float64,43.51,2,"[0.0, 1.0]"
2,alcohol_binge_any_30d,float64,36.24,2,"[0.0, 1.0]"
3,alcohol_ever_daily_heavy,float64,21.77,2,"[0.0, 1.0]"
4,alcohol_drinks_per_day_bin,category,35.71,4,"[up_to_1, more_than_4, up_to_2, up_to_4]"
5,alcohol_drinks_per_day_ge_2,float64,35.71,2,"[0.0, 1.0]"
6,alcohol_drinks_per_day_ge_4,float64,35.71,2,"[0.0, 1.0]"
7,alcohol_binge_days_12m_bin,category,43.51,3,"[0, 2_to_12, 1]"
8,alcohol_binge_monthly_or_more,float64,43.51,1,[0.0]
9,alcohol_any_risk_signal,float64,19.44,2,"[0.0, 1.0]"


## Model

In [23]:
med_count_col = None

for c in MED_COUNT_CANDIDATES:
    if c in df.columns:
        med_count_col = c
        break

if med_count_col is None:
    medication_cols = [c for c in df.columns if c.lower().startswith("medication_")]
    if medication_cols:
        df["med_count"] = (
            df[medication_cols]
            .replace("", np.nan)
            .notna()
            .sum(axis=1)
        )
        med_count_col = "med_count"

print("Using med count column:", med_count_col)

if med_count_col is None:
    print("WARNING: No med_count-like feature found or created.")
else:
    print(df[med_count_col].describe())

Using med count column: med_count
count    7437.000000
mean        2.114966
std         2.295032
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        22.000000
Name: med_count, dtype: float64


In [24]:
overview = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_pct": df.isna().mean().values * 100,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns]
}).sort_values(["missing_pct", "n_unique"], ascending=[False, True])

overview.head(30)

,feature,dtype,missing_pct,n_unique
166,mcq149___menstrual_periods_started_yet?,float64,100.000000,0
167,mcq151___age_in_years_at_first_menstrual_period,float64,100.000000,0
294,smq621___cigarettes_smoked_in_entire_life,float64,100.000000,0
295,smd630___age_first_smoked_whole_cigarette,float64,100.000000,0
434,LBXIGG_cytomegalovirus_cmv_igg,float64,100.000000,0
435,LBXIGM_cytomegalovirus_cmv_igm,float64,100.000000,0
436,LBXIGGA_cytomegalovirus_cmv_igg_avidity,float64,100.000000,0
357,medication_21,str,99.973107,2
358,medication_22,str,99.973107,2
356,medication_20,str,99.959661,3


In [36]:
LEAKAGE_COLS = [
    TARGET,
    "LBXSNASI_sodium_mmol_l",
    "LBXSKSI_potassium_mmol_l",
    "LBXSCA_total_calcium_mg_dl",
    "rxd_disease_list",
]

QUESTIONNAIRE_PREFIXES = (
    "slq", "sld", "dpq", "paq", "huq", "mcq", "bpq", "kiq", "cdq", "smq", "alq",
    "whq", "rhq"
)

EXCLUDE_PATTERNS = [
    r"^seqn$",
    r"cluster",
    r"stratum",
    r"weight",
    r"sample_weight",
    r"survey_",
    r"^rxd",
    r"medication",
    r"drug",
    r"icd",
    r"diagnosis",
    r"lab",
    r"exam",
    r"fasting",
    r"liver_",
    r"bmi$",
    r"waist",
    r"hip",
    r"height",
    r"weight_kg",
    r"cholesterol",
    r"creatin",
    r"ferritin",
    r"vitamin",
    r"iron",
    r"protein",
    r"carbs",
    r"fat$",
    r"calories",
    r"pulse_",
    r"fatigue_binary",
    r"fatigue_score",
    r"sodium",
    r"potassium",
    r"calcium",
    r"chloride",
    r"magnesium",
    r"phosph",
    r"osmol",
]

MANUAL_INCLUDE_COLS = []
if med_count_col is not None:
    MANUAL_INCLUDE_COLS.append(med_count_col)

# Alkohol FE Features
ALCOHOL_MODEL_FEATURES = [
    "alcohol_ever_yes",
    "alcohol_drinks_per_day_ge_2",
    "alcohol_ever_daily_heavy",
    "alcohol_any_risk_signal",
]

MANUAL_DROP_COLS = [
    "bpq040a___taking_prescription_for_hypertension",
    "bpq050a___now_taking_prescribed_medicine_for_hbp",
    "bpq100d___now_taking_prescribed_medicine",
    "kiq025___received_dialysis_in_past_12_months?",
    "mcq050___emergency_care_visit_for_asthma/past_yr",
    "mcq053___taking_treatment_for_anemia/past_3_mos",
    "mcq540___ever_seen_a_dr_about_this_pain",
    "mcq025___age_when_first_had_asthma",
    "mcq195___which_type_of_arthritis_was_it?",
    "mcq170m___still_have_thyroid_problem",
    "mcq530___where_was_the_most_uncomfortable_pain",
    "mcq570___age_when_1st_had_gallbladder_surgery?",
    "mcq230a___what_kind_of_cancer_first_mention",
    "mcq230b___what_kind_of_cancer_second_mention",
    "mcq230c___what_kind_of_cancer_third_mention",
    "mcq149___menstrual_periods_started_yet?",
    "mcq151___age_in_years_at_first_menstrual_period",
    "huq030___routine_place_to_go_for_healthcare",
    "huq051___#times_receive_healthcare_over_past_year",
    "huq071___overnight_hospital_patient_in_last_year",
    "huq090___seen_mental_health_professional/past_yr",
    "alq270___#_times_4_5_drinks_in_2hrs/past_12_mos",
    "alq280___#_times_8+_drinks_in_1_day/past_12_mos",
    "alq290___#_times_12+_drinks_in_1_day/past_12_mos",
    "alq170___past_30_days_#_times_4_5_drinks_on_an_oc",
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol",
    "alq121___past_12_mo_how_often_drink_alcoholic_bev",
    "alq130___avg_#_alcoholic_drinks/day___past_12_mos",
    "alq142___#_days_have_4_or_5_drinks/past_12_mos",
    "alq151___ever_have_4/5_or_more_drinks_every_day?",
    "kiq010___how_much_urine_lose_each_time?",
    "kiq042___leak_urine_during_physical_activities?",
    "kiq430___how_frequently_does_this_occur?",
    "kiq450___how_frequently_does_this_occur?",
    "kiq470___how_frequently_does_this_occur?",
    "kiq050___how_much_did_urine_leakage_bother_you?",
    "kiq052___how_much_were_daily_activities_affected?",
    "cdq002___sp_get_it_walking_uphill_or_in_a_hurry",
    "cdq003___during_an_ordinary_pace_on_level_ground",
    "cdq004___if_so_does_sp_continue_or_slow_down",
    "cdq005___does_standing_relieve_pain/discomfort",
    "cdq006___how_soon_is_the_pain_relieved",
    "cdq009a___pain_in_right_arm",
    "cdq009b___pain_in_right_chest",
    "cdq009c___pain_in_neck",
    "cdq009d___pain_in_upper_sternum",
    "cdq009e___pain_in_lower_sternum",
    "cdq009f___pain_in_left_chest",
    "cdq009g___pain_in_left_arm",
    "cdq008___severe_pain_in_chest_more_than_half_hour",

    # doctor advice / behavior change aftermath
    "mcq366b___doctor_told_to_increase_exercise",
    "mcq366c___doctor_told_to_reduce_salt",
    "mcq366d___doctor_told_to_reduce_fat_in_diet",
    "mcq371b___doing_increasing_exercise",
    "mcq371c___doing_reducing_salt",
    "mcq371d___doing_reducing_fat_in_diet",

    # reproductive / pregnancy specific: likely too specific and weakly aligned
    "rhq010___age_when_first_menstrual_period_occurred",
    "rhq020___age_range_at_first_menstrual_period",
    "rhq031___had_regular_periods_in_past_12_months",
    "rhq060___age_at_last_menstrual_period",
    "rhq070___age_range_at_last_menstrual_period",
    "rhq074___tried_for_a_year_to_become_pregnant?",
    "rhq076___seen_a_dr_b/c_unable_to_become_pregnant?",
    "rhq078___ever_treated_for_a_pelvic_infection/pid?",
    "rhq131___ever_been_pregnant?",
    "rhq160___how_many_times_have_been_pregnant?",
    "rhq162___during_pregnancy,_told_you_have_diabetes",
    "rhq171___how_many_deliveries_live_birth_result?",
    "rhq172___any_babies_weigh_9_lbs_or_more?",
    "rhq197___how_many_months_ago_have_baby?",
    "rhq200___now_breastfeeding_a_child?",
    "rhq305___had_both_ovaries_removed?",
    "rhq332___age_when_both_ovaries_removed",
    "rhq540___ever_use_female_hormones?",
    "rhq542a___hormone_pills_used",
    "rhq542b___hormone_patches_used",
    "rhq542c___hormone_cream/suppository/injection_used",
    "rhq554___use_hormone_pills_w/estrogen_only",
    "rhq570___used_estrogen/progestin_combo_pills",

    # family history less aligned for electrolyte imbalance
    "mcq300b___close_relative_had_asthma",
    "mcq300c___close_relative_had_diabetes",
    "mcq300a___close_relative_had_heart_attack",

    # old raw sleep fields if you plan to use engineered sleep features instead
    "slq300___usual_sleep_time_on_weekdays_or_workdays",
    "slq310___usual_wake_time_on_weekdays_or_workdays",
    "sld012___sleep_hours___weekdays_or_workdays",
    "slq320___usual_sleep_time_on_weekends",

    # morbidity marker
    "slq050___ever_told_doctor_had_trouble_sleeping?",
    "mcq160b___ever_told_you_had_congestive_heart_failure",
    "mcq160c___ever_told_you_had_coronary_heart_disease",
    "mcq160d___ever_told_you_had_angina",
    "mcq160e___ever_told_you_had_heart_attack",
    "mcq160f___ever_told_you_had_stroke",
    "mcq220___ever_told_you_had_cancer_or_malignancy",

    # too close to sleeping disfunction
    "slq330___usual_wake_time_on_weekends",
    "sld013___sleep_hours___weekends",
    "slq040___how_often_do_you_snort_or_stop_breathing",

]

In [37]:
def has_allowed_prefix(col: str, prefixes=QUESTIONNAIRE_PREFIXES) -> bool:
    col_low = col.lower()
    return col_low.startswith(prefixes)

def matches_any_pattern(col: str, patterns) -> bool:
    col_low = col.lower()
    return any(re.search(p, col_low) for p in patterns)

def is_binary_series(s: pd.Series) -> bool:
    vals = set(pd.Series(s.dropna()).unique())
    return vals.issubset({0, 1}) and len(vals) <= 2

def get_example_values(s: pd.Series, n=5):
    vals = s.dropna().unique()[:n]
    return list(vals)

def cramers_v(x: pd.Series, y: pd.Series) -> float:
    table = pd.crosstab(x, y)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()
    r, k = table.shape
    denom = min(k - 1, r - 1)
    if denom == 0 or n == 0:
        return np.nan
    return np.sqrt((chi2 / n) / denom)

def safe_pointbiserial(x: pd.Series, y: pd.Series):
    valid = x.notna() & y.notna()
    if valid.sum() < 30:
        return np.nan, np.nan
    try:
        r, p = pointbiserialr(x[valid], y[valid])
        return r, p
    except Exception:
        return np.nan, np.nan

def infer_feature_type(series: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(series):
        nunique = series.nunique(dropna=True)
        if nunique <= 10:
            return "categorical"
        return "numeric"
    return "categorical"

In [38]:
candidate_cols = []

for col in df.columns:
    if col == TARGET:
        continue
    if col in LEAKAGE_COLS:
        continue
    if col in MANUAL_DROP_COLS:
        continue

    if col in MANUAL_INCLUDE_COLS:
        candidate_cols.append(col)
        continue

    if not has_allowed_prefix(col):
        continue
    if matches_any_pattern(col, EXCLUDE_PATTERNS):
        continue

    candidate_cols.append(col)

candidate_cols = list(dict.fromkeys(candidate_cols))

print("Number of initial candidate features:", len(candidate_cols))
print(candidate_cols[:80])

Number of initial candidate features: 43
['bpq020___ever_told_you_had_high_blood_pressure', 'bpq030___told_had_high_blood_pressure___2+_times', 'cdq001___sp_ever_had_pain_or_discomfort_in_chest', 'cdq010___shortness_of_breath_on_stairs/inclines', 'dpq040___feeling_tired_or_having_little_energy', 'huq010___general_health_condition', 'kiq022___ever_told_you_had_weak/failing_kidneys?', 'kiq026___ever_had_kidney_stones?', 'kiq029___pass_kidney_stone_in_past_12_months?', 'kiq005___how_often_have_urinary_leakage?', 'kiq044___urinated_before_reaching_the_toilet?', 'kiq046___leak_urine_during_nonphysical_activities', 'kiq480___how_many_times_urinate_in_night?', 'mcq010___ever_been_told_you_have_asthma', 'mcq035___still_have_asthma', 'mcq040___had_asthma_attack_in_past_year', 'mcq092___ever_receive_blood_transfusion', 'mcq160a___ever_told_you_had_arthritis', 'mcq160m___ever_told_you_had_thyroid_problem', 'mcq160p___ever_told_you_had_copd_emphysema', 'mcq520___abdominal_pain_during_past_12_month

In [39]:
audit_rows = []

for col in candidate_cols:
    s = df[col]
    audit_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "n_unique": s.nunique(dropna=True),
        "example_values": str(get_example_values(s, n=5)),
        "is_binary_like": is_binary_series(s),
        "keep_for_model": True,
        "drop_reason": ""
    })

audit_df = pd.DataFrame(audit_rows).sort_values(
    ["missing_pct", "n_unique", "feature"],
    ascending=[False, True, True]
)

audit_df.head(100)

,feature,dtype,missing_pct,n_unique,example_values,is_binary_like,keep_for_model,drop_reason
40,smq621___cigarettes_smoked_in_entire_life,float64,100.00,0,[],True,True,
8,kiq029___pass_kidney_stone_in_past_12_months?,float64,91.93,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False,True,
15,mcq040___had_asthma_attack_in_past_year,float64,90.13,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False,True,
38,smq078___how_soon_after_waking_do_you_smoke,float64,85.16,8,"[np.float64(2.0), np.float64(1.0), np.float64(...",False,True,
14,mcq035___still_have_asthma,float64,83.19,4,"[np.float64(1.0), np.float64(2.0), np.float64(...",False,True,
37,smq050u___unit_of_measure_(day/week/month/year),float64,82.37,4,"[np.float64(3.0), np.float64(4.0), np.float64(...",False,True,
36,smq050q___how_long_since_quit_smoking_cigarettes,float64,82.32,50,"[np.float64(8.0), np.float64(1.0), np.float64(...",False,True,
39,smq670___tried_to_quit_smoking,float64,80.26,2,"[np.float64(1.0), np.float64(2.0)]",False,True,
28,paq640___number_of_days_walk_or_bicycle,float64,74.33,8,"[np.float64(4.0), np.float64(5.0), np.float64(...",False,True,
24,paq610___number_of_days_vigorous_work,float64,72.00,8,"[np.float64(5.0), np.float64(3.0), np.float64(...",False,True,


In [40]:
MAX_MISSING_PCT = 60.0
MIN_NON_NULL = 500
MIN_VARIANCE_UNIQUE = 2

filtered_audit_df = audit_df.copy()

filtered_audit_df.loc[filtered_audit_df["missing_pct"] > MAX_MISSING_PCT, "keep_for_model"] = False
filtered_audit_df.loc[filtered_audit_df["missing_pct"] > MAX_MISSING_PCT, "drop_reason"] = "too_missing"

for idx, row in filtered_audit_df.iterrows():
    col = row["feature"]
    non_null = df[col].notna().sum()
    if non_null < MIN_NON_NULL:
        filtered_audit_df.at[idx, "keep_for_model"] = False
        filtered_audit_df.at[idx, "drop_reason"] = "too_few_non_null"
    elif row["n_unique"] < MIN_VARIANCE_UNIQUE:
        filtered_audit_df.at[idx, "keep_for_model"] = False
        filtered_audit_df.at[idx, "drop_reason"] = "constant_or_near_constant"

filtered_audit_df["keep_for_model"].value_counts()

keep_for_model
True     30
False    13
Name: count, dtype: int64

In [42]:
MANUAL_DROP_AFTER_AUDIT = [
    # Beispiele - nach Audit anpassen
    # "huq051___#times_receive_healthcare_over_past_year",
    # "huq030___routine_place_to_go_for_healthcare",
]

In [43]:
filtered_audit_df.loc[
    filtered_audit_df["feature"].isin(MANUAL_DROP_AFTER_AUDIT),
    ["keep_for_model", "drop_reason"]
] = [False, "manual_drop"]

final_features = filtered_audit_df.loc[filtered_audit_df["keep_for_model"], "feature"].tolist()

print("Final feature count:", len(final_features))
print(final_features[:80])

Final feature count: 30
['paq670___days_moderate_recreational_activities', 'paq625___number_of_days_moderate_work', 'cdq001___sp_ever_had_pain_or_discomfort_in_chest', 'cdq010___shortness_of_breath_on_stairs/inclines', 'kiq046___leak_urine_during_nonphysical_activities', 'kiq480___how_many_times_urinate_in_night?', 'kiq044___urinated_before_reaching_the_toilet?', 'kiq005___how_often_have_urinary_leakage?', 'dpq040___feeling_tired_or_having_little_energy', 'kiq022___ever_told_you_had_weak/failing_kidneys?', 'kiq026___ever_had_kidney_stones?', 'mcq160a___ever_told_you_had_arthritis', 'mcq160p___ever_told_you_had_copd_emphysema', 'mcq520___abdominal_pain_during_past_12_months?', 'mcq550___has_dr_ever_said_you_have_gallstones', 'mcq560___ever_had_gallbladder_surgery?', 'mcq160m___ever_told_you_had_thyroid_problem', 'paq650___vigorous_recreational_activities', 'paq665___moderate_recreational_activities', 'bpq020___ever_told_you_had_high_blood_pressure', 'mcq010___ever_been_told_you_have_ast

In [44]:
final_audit_path = OUTPUT_DIR / "electrolyte_imbalance_feature_audit_final.csv"
filtered_audit_df.to_csv(final_audit_path, index=False)
print("Saved:", final_audit_path.resolve())

Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_electrolyte_imbalance\electrolyte_imbalance_feature_audit_final.csv


In [45]:
model_df = df[[TARGET] + final_features].copy()

print("Model dataframe shape:", model_df.shape)
print("Target missing:", model_df[TARGET].isna().sum())

X = model_df[final_features].copy()
y = model_df[TARGET].copy()

mask = y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True).round(4))

Model dataframe shape: (7437, 31)
Target missing: 0
Train shape: (5949, 30)
Test shape: (1488, 30)
Train target distribution:
electrolyte_imbalance
0    0.9222
1    0.0778
Name: proportion, dtype: float64


In [46]:
univariate_rows = []

for col in final_features:
    s = X_train[col]
    feature_type = infer_feature_type(s)

    row = {
        "feature": col,
        "feature_type": feature_type,
        "missing_pct_train": round(s.isna().mean() * 100, 2),
        "n_unique_train": s.nunique(dropna=True),
        "association_metric": None,
        "association_value": np.nan,
        "abs_association": np.nan,
        "p_value": np.nan,
        "n_train_non_null": int(s.notna().sum())
    }

    if feature_type == "categorical":
        valid = s.notna() & y_train.notna()
        if valid.sum() >= 30:
            try:
                cv = cramers_v(s[valid], y_train[valid])
                table = pd.crosstab(s[valid], y_train[valid])
                p_val = chi2_contingency(table)[1] if table.shape[0] > 1 and table.shape[1] > 1 else np.nan
                row["association_metric"] = "cramers_v"
                row["association_value"] = cv
                row["abs_association"] = abs(cv) if pd.notna(cv) else np.nan
                row["p_value"] = p_val
            except Exception:
                pass
    else:
        r, p = safe_pointbiserial(s, y_train)
        row["association_metric"] = "pointbiserial"
        row["association_value"] = r
        row["abs_association"] = abs(r) if pd.notna(r) else np.nan
        row["p_value"] = p

    univariate_rows.append(row)

univariate_df = pd.DataFrame(univariate_rows).sort_values(
    ["abs_association", "p_value"],
    ascending=[False, True]
)

univariate_df.head(50)

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null
29,med_count,numeric,0.00,21,pointbiserial,0.129593,0.129593,1.061670e-23,5949
19,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.116671,0.116671,2.603849e-18,5949
5,kiq480___how_many_times_urinate_in_night?,categorical,17.92,8,cramers_v,0.095109,0.095109,1.980451e-07,4883
0,paq670___days_moderate_recreational_activities,categorical,57.40,8,cramers_v,0.082319,0.082319,1.632425e-02,2534
11,mcq160a___ever_told_you_had_arthritis,categorical,5.98,3,cramers_v,0.077041,0.077041,6.188193e-08,5593
28,huq010___general_health_condition,categorical,0.00,7,cramers_v,0.074960,0.074960,8.676980e-06,5949
3,cdq010___shortness_of_breath_on_stairs/inclines,categorical,43.60,3,cramers_v,0.067029,0.067029,5.330863e-04,3355
17,paq650___vigorous_recreational_activities,categorical,0.00,2,cramers_v,0.064761,0.064761,5.883427e-07,5949
1,paq625___number_of_days_moderate_work,categorical,54.40,8,cramers_v,0.062374,0.062374,1.592500e-01,2713
9,kiq022___ever_told_you_had_weak/failing_kidneys?,categorical,5.98,3,cramers_v,0.058777,0.058777,6.371589e-05,5593


In [47]:
univariate_path = OUTPUT_DIR / "electrolyte_imbalance_univariate_ranking.csv"
univariate_df.to_csv(univariate_path, index=False)
print("Saved:", univariate_path.resolve())

Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_electrolyte_imbalance\electrolyte_imbalance_univariate_ranking.csv


In [48]:
numeric_features = []
categorical_features = []

for col in final_features:
    if infer_feature_type(X_train[col]) == "numeric":
        numeric_features.append(col)
    else:
        categorical_features.append(col)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 1
Categorical features: 29


In [49]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

In [50]:
baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

print("Baseline precision:", precision_score(y_test, baseline_pred, zero_division=0))
print("Baseline recall:", recall_score(y_test, baseline_pred, zero_division=0))
print("Baseline f1:", f1_score(y_test, baseline_pred, zero_division=0))

Baseline precision: 0.0
Baseline recall: 0.0
Baseline f1: 0.0


In [51]:
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [52]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

logreg_cv = cross_validate(logreg_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv = cross_validate(rf_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary = pd.DataFrame({
    "model": ["logreg", "random_forest"],
    "roc_auc_mean": [logreg_cv["test_roc_auc"].mean(), rf_cv["test_roc_auc"].mean()],
    "avg_precision_mean": [logreg_cv["test_avg_precision"].mean(), rf_cv["test_avg_precision"].mean()],
    "f1_mean": [logreg_cv["test_f1"].mean(), rf_cv["test_f1"].mean()],
    "precision_mean": [logreg_cv["test_precision"].mean(), rf_cv["test_precision"].mean()],
    "recall_mean": [logreg_cv["test_recall"].mean(), rf_cv["test_recall"].mean()],
}).sort_values("avg_precision_mean", ascending=False)

cv_summary



,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
0,logreg,0.618493,0.125857,0.184108,0.113338,0.490369
1,random_forest,0.599808,0.114004,0.062599,0.133725,0.041094


In [53]:
best_pipeline = logreg_pipeline
best_model_name = "logreg"

best_pipeline.fit(X_train, y_train)

test_proba = best_pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.50).astype(int)

print("Best model:", best_model_name)
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))
print("Average Precision:", round(average_precision_score(y_test, test_proba), 4))
print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, test_pred, zero_division=0), 4))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))
print()
print(classification_report(y_test, test_pred, zero_division=0))

Best model: logreg
ROC-AUC: 0.6463
Average Precision: 0.1447
Precision: 0.1284
Recall: 0.5345
F1: 0.207

Confusion Matrix:
[[951 421]
 [ 54  62]]

              precision    recall  f1-score   support

           0       0.95      0.69      0.80      1372
           1       0.13      0.53      0.21       116

    accuracy                           0.68      1488
   macro avg       0.54      0.61      0.50      1488
weighted avg       0.88      0.68      0.75      1488



In [54]:
perm = permutation_importance(
    best_pipeline,
    X_test,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df.head(30)

,feature,importance_mean,importance_std
19,bpq020___ever_told_you_had_high_blood_pressure,0.016406,0.006480
29,med_count,0.010719,0.007330
5,kiq480___how_many_times_urinate_in_night?,0.009140,0.007126
17,paq650___vigorous_recreational_activities,0.007691,0.003829
13,mcq520___abdominal_pain_during_past_12_months?,0.003068,0.001872
8,dpq040___feeling_tired_or_having_little_energy,0.002808,0.003689
11,mcq160a___ever_told_you_had_arthritis,0.002786,0.001854
10,kiq026___ever_had_kidney_stones?,0.002701,0.000997
7,kiq005___how_often_have_urinary_leakage?,0.001873,0.002450
9,kiq022___ever_told_you_had_weak/failing_kidneys?,0.001659,0.001483


In [55]:
combined_df = (
    univariate_df.merge(perm_df, on="feature", how="left")
    .merge(filtered_audit_df[["feature", "missing_pct", "n_unique", "drop_reason"]], on="feature", how="left")
)

combined_df = combined_df.sort_values(
    ["importance_mean", "abs_association"],
    ascending=[False, False]
)

combined_df.head(40)

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null,importance_mean,importance_std,missing_pct,n_unique,drop_reason
1,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.116671,0.116671,2.603849e-18,5949,0.016406,0.006480,0.00,3,
0,med_count,numeric,0.00,21,pointbiserial,0.129593,0.129593,1.061670e-23,5949,0.010719,0.007330,0.00,21,
2,kiq480___how_many_times_urinate_in_night?,categorical,17.92,8,cramers_v,0.095109,0.095109,1.980451e-07,4883,0.009140,0.007126,18.14,8,
7,paq650___vigorous_recreational_activities,categorical,0.00,2,cramers_v,0.064761,0.064761,5.883427e-07,5949,0.007691,0.003829,0.00,2,
27,mcq520___abdominal_pain_during_past_12_months?,categorical,5.98,3,cramers_v,0.011242,0.011242,7.022589e-01,5593,0.003068,0.001872,6.20,3,
13,dpq040___feeling_tired_or_having_little_energy,categorical,13.08,6,cramers_v,0.045179,0.045179,6.095914e-02,5171,0.002808,0.003689,13.03,6,
4,mcq160a___ever_told_you_had_arthritis,categorical,5.98,3,cramers_v,0.077041,0.077041,6.188193e-08,5593,0.002786,0.001854,6.20,3,
17,kiq026___ever_had_kidney_stones?,categorical,5.98,3,cramers_v,0.037789,0.037789,1.843674e-02,5593,0.002701,0.000997,6.20,3,
11,kiq005___how_often_have_urinary_leakage?,categorical,17.89,7,cramers_v,0.050456,0.050456,5.291763e-02,4885,0.001873,0.002450,18.10,7,
9,kiq022___ever_told_you_had_weak/failing_kidneys?,categorical,5.98,3,cramers_v,0.058777,0.058777,6.371589e-05,5593,0.001659,0.001483,6.20,3,


In [56]:
combined_df = (
    univariate_df.merge(perm_df, on="feature", how="left")
    .merge(filtered_audit_df[["feature", "missing_pct", "n_unique", "drop_reason"]], on="feature", how="left")
)

combined_df = combined_df.sort_values(
    ["importance_mean", "abs_association"],
    ascending=[False, False]
)

combined_df.head(40)

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null,importance_mean,importance_std,missing_pct,n_unique,drop_reason
1,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.116671,0.116671,2.603849e-18,5949,0.016406,0.006480,0.00,3,
0,med_count,numeric,0.00,21,pointbiserial,0.129593,0.129593,1.061670e-23,5949,0.010719,0.007330,0.00,21,
2,kiq480___how_many_times_urinate_in_night?,categorical,17.92,8,cramers_v,0.095109,0.095109,1.980451e-07,4883,0.009140,0.007126,18.14,8,
7,paq650___vigorous_recreational_activities,categorical,0.00,2,cramers_v,0.064761,0.064761,5.883427e-07,5949,0.007691,0.003829,0.00,2,
27,mcq520___abdominal_pain_during_past_12_months?,categorical,5.98,3,cramers_v,0.011242,0.011242,7.022589e-01,5593,0.003068,0.001872,6.20,3,
13,dpq040___feeling_tired_or_having_little_energy,categorical,13.08,6,cramers_v,0.045179,0.045179,6.095914e-02,5171,0.002808,0.003689,13.03,6,
4,mcq160a___ever_told_you_had_arthritis,categorical,5.98,3,cramers_v,0.077041,0.077041,6.188193e-08,5593,0.002786,0.001854,6.20,3,
17,kiq026___ever_had_kidney_stones?,categorical,5.98,3,cramers_v,0.037789,0.037789,1.843674e-02,5593,0.002701,0.000997,6.20,3,
11,kiq005___how_often_have_urinary_leakage?,categorical,17.89,7,cramers_v,0.050456,0.050456,5.291763e-02,4885,0.001873,0.002450,18.10,7,
9,kiq022___ever_told_you_had_weak/failing_kidneys?,categorical,5.98,3,cramers_v,0.058777,0.058777,6.371589e-05,5593,0.001659,0.001483,6.20,3,


# EI-focused features

In [57]:
EI_FOCUSED_FEATURES = [

    # strongest signals
    "bpq020___ever_told_you_had_high_blood_pressure",
    "med_count",
    "kiq480___how_many_times_urinate_in_night?",
    "paq650___vigorous_recreational_activities",

    # strong medical plausibility
    "mcq160a___ever_told_you_had_arthritis",
    "kiq026___ever_had_kidney_stones?",
    "kiq022___ever_told_you_had_weak/failing_kidneys?",
    "dpq040___feeling_tired_or_having_little_energy",

    # optional extras
    "mcq160p___ever_told_you_had_copd_emphysema",
    "kiq005___how_often_have_urinary_leakage?",
]

In [58]:
# Electrolyte imbalance focused only run

EI_FOCUSED_FEATURES = [
    "bpq020___ever_told_you_had_high_blood_pressure",
    "med_count",
    "kiq480___how_many_times_urinate_in_night?",
    "paq650___vigorous_recreational_activities",
    "mcq160a___ever_told_you_had_arthritis",
    "kiq026___ever_had_kidney_stones?",
    "kiq022___ever_told_you_had_weak/failing_kidneys?",
    "dpq040___feeling_tired_or_having_little_energy",
    "mcq160p___ever_told_you_had_copd_emphysema",
    "kiq005___how_often_have_urinary_leakage?",
]

ei_focused_available = [c for c in EI_FOCUSED_FEATURES if c in df.columns]
ei_focused_missing = [c for c in EI_FOCUSED_FEATURES if c not in df.columns]

print("Available EI-focused features:", len(ei_focused_available))
print(ei_focused_available)
print()
print("Missing requested features:", len(ei_focused_missing))
print(ei_focused_missing)

Available EI-focused features: 10
['bpq020___ever_told_you_had_high_blood_pressure', 'med_count', 'kiq480___how_many_times_urinate_in_night?', 'paq650___vigorous_recreational_activities', 'mcq160a___ever_told_you_had_arthritis', 'kiq026___ever_had_kidney_stones?', 'kiq022___ever_told_you_had_weak/failing_kidneys?', 'dpq040___feeling_tired_or_having_little_energy', 'mcq160p___ever_told_you_had_copd_emphysema', 'kiq005___how_often_have_urinary_leakage?']

Missing requested features: 0
[]


In [59]:
# Electrolyte imbalance focused only run

EI_FOCUSED_FEATURES = [
    "bpq020___ever_told_you_had_high_blood_pressure",
    "med_count",
    "kiq480___how_many_times_urinate_in_night?",
    "paq650___vigorous_recreational_activities",
    "mcq160a___ever_told_you_had_arthritis",
    "kiq026___ever_had_kidney_stones?",
    "kiq022___ever_told_you_had_weak/failing_kidneys?",
    "dpq040___feeling_tired_or_having_little_energy",
    "mcq160p___ever_told_you_had_copd_emphysema",
    "kiq005___how_often_have_urinary_leakage?",
]

ei_focused_available = [c for c in EI_FOCUSED_FEATURES if c in df.columns]
ei_focused_missing = [c for c in EI_FOCUSED_FEATURES if c not in df.columns]

print("Available EI-focused features:", len(ei_focused_available))
print(ei_focused_available)
print()
print("Missing requested features:", len(ei_focused_missing))
print(ei_focused_missing)

Available EI-focused features: 10
['bpq020___ever_told_you_had_high_blood_pressure', 'med_count', 'kiq480___how_many_times_urinate_in_night?', 'paq650___vigorous_recreational_activities', 'mcq160a___ever_told_you_had_arthritis', 'kiq026___ever_had_kidney_stones?', 'kiq022___ever_told_you_had_weak/failing_kidneys?', 'dpq040___feeling_tired_or_having_little_energy', 'mcq160p___ever_told_you_had_copd_emphysema', 'kiq005___how_often_have_urinary_leakage?']

Missing requested features: 0
[]


In [60]:
ei_focus_audit_rows = []

for col in ei_focused_available:
    s = df[col]
    ei_focus_audit_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "n_unique": s.nunique(dropna=True),
        "example_values": str(get_example_values(s, n=8)),
        "is_binary_like": is_binary_series(s),
    })

ei_focus_audit_df = pd.DataFrame(ei_focus_audit_rows).sort_values(
    ["missing_pct", "n_unique", "feature"],
    ascending=[False, True, True]
)

ei_focus_audit_df

,feature,dtype,missing_pct,n_unique,example_values,is_binary_like
2,kiq480___how_many_times_urinate_in_night?,float64,18.14,8,"[np.float64(0.0), np.float64(2.0), np.float64(...",False
9,kiq005___how_often_have_urinary_leakage?,float64,18.10,7,"[np.float64(1.0), np.float64(2.0), np.float64(...",False
7,dpq040___feeling_tired_or_having_little_energy,float64,13.03,6,"[np.float64(0.0), np.float64(2.0), np.float64(...",False
6,kiq022___ever_told_you_had_weak/failing_kidneys?,float64,6.20,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
5,kiq026___ever_had_kidney_stones?,float64,6.20,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
4,mcq160a___ever_told_you_had_arthritis,float64,6.20,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
8,mcq160p___ever_told_you_had_copd_emphysema,float64,6.20,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
3,paq650___vigorous_recreational_activities,float64,0.00,2,"[np.float64(1.0), np.float64(2.0)]",False
0,bpq020___ever_told_you_had_high_blood_pressure,float64,0.00,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
1,med_count,int64,0.00,21,"[np.int64(1), np.int64(3), np.int64(6), np.int...",False


In [62]:
EI_FOCUS_MANUAL_DROP = [
    # optional later, e.g.
    # "mcq160p___ever_told_you_had_copd_emphysema",
    # "kiq005___how_often_have_urinary_leakage?",
]

EI_FOCUS_MAX_MISSING_PCT = 60.0

ei_focused_final_features = [
    c for c in ei_focused_available
    if c not in EI_FOCUS_MANUAL_DROP
    and df[c].isna().mean() * 100 <= EI_FOCUS_MAX_MISSING_PCT
]

print("Final EI-focused features:", len(ei_focused_final_features))
print(ei_focused_final_features)

Final EI-focused features: 10
['bpq020___ever_told_you_had_high_blood_pressure', 'med_count', 'kiq480___how_many_times_urinate_in_night?', 'paq650___vigorous_recreational_activities', 'mcq160a___ever_told_you_had_arthritis', 'kiq026___ever_had_kidney_stones?', 'kiq022___ever_told_you_had_weak/failing_kidneys?', 'dpq040___feeling_tired_or_having_little_energy', 'mcq160p___ever_told_you_had_copd_emphysema', 'kiq005___how_often_have_urinary_leakage?']


In [63]:
ei_focus_df = df[[TARGET] + ei_focused_final_features].copy()

mask = ei_focus_df[TARGET].notna()
X_ei = ei_focus_df.loc[mask, ei_focused_final_features].copy()
y_ei = ei_focus_df.loc[mask, TARGET].astype(int).copy()

X_train_ei, X_test_ei, y_train_ei, y_test_ei = train_test_split(
    X_ei, y_ei,
    test_size=TEST_SIZE,
    stratify=y_ei,
    random_state=RANDOM_STATE
)

print("EI-focused train shape:", X_train_ei.shape)
print("EI-focused test shape:", X_test_ei.shape)
print("Train target distribution:")
print(y_train_ei.value_counts(normalize=True).round(4))

EI-focused train shape: (5949, 10)
EI-focused test shape: (1488, 10)
Train target distribution:
electrolyte_imbalance
0    0.9222
1    0.0778
Name: proportion, dtype: float64


In [64]:
ei_univariate_rows = []

for col in ei_focused_final_features:
    s = X_train_ei[col]
    feature_type = infer_feature_type(s)

    row = {
        "feature": col,
        "feature_type": feature_type,
        "missing_pct_train": round(s.isna().mean() * 100, 2),
        "n_unique_train": s.nunique(dropna=True),
        "association_metric": None,
        "association_value": np.nan,
        "abs_association": np.nan,
        "p_value": np.nan,
        "n_train_non_null": int(s.notna().sum())
    }

    if feature_type == "categorical":
        valid = s.notna() & y_train_ei.notna()
        if valid.sum() >= 30:
            try:
                cv = cramers_v(s[valid], y_train_ei[valid])
                table = pd.crosstab(s[valid], y_train_ei[valid])
                p_val = chi2_contingency(table)[1] if table.shape[0] > 1 and table.shape[1] > 1 else np.nan
                row["association_metric"] = "cramers_v"
                row["association_value"] = cv
                row["abs_association"] = abs(cv) if pd.notna(cv) else np.nan
                row["p_value"] = p_val
            except Exception:
                pass
    else:
        r, p = safe_pointbiserial(s, y_train_ei)
        row["association_metric"] = "pointbiserial"
        row["association_value"] = r
        row["abs_association"] = abs(r) if pd.notna(r) else np.nan
        row["p_value"] = p

    ei_univariate_rows.append(row)

ei_univariate_df = pd.DataFrame(ei_univariate_rows).sort_values(
    ["abs_association", "p_value"],
    ascending=[False, True]
)

ei_univariate_df

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null
1,med_count,numeric,0.00,21,pointbiserial,0.129593,0.129593,1.061670e-23,5949
0,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.116671,0.116671,2.603849e-18,5949
2,kiq480___how_many_times_urinate_in_night?,categorical,17.92,8,cramers_v,0.095109,0.095109,1.980451e-07,4883
4,mcq160a___ever_told_you_had_arthritis,categorical,5.98,3,cramers_v,0.077041,0.077041,6.188193e-08,5593
3,paq650___vigorous_recreational_activities,categorical,0.00,2,cramers_v,0.064761,0.064761,5.883427e-07,5949
6,kiq022___ever_told_you_had_weak/failing_kidneys?,categorical,5.98,3,cramers_v,0.058777,0.058777,6.371589e-05,5593
8,mcq160p___ever_told_you_had_copd_emphysema,categorical,5.98,3,cramers_v,0.053428,0.053428,3.413038e-04,5593
9,kiq005___how_often_have_urinary_leakage?,categorical,17.89,7,cramers_v,0.050456,0.050456,5.291763e-02,4885
7,dpq040___feeling_tired_or_having_little_energy,categorical,13.08,6,cramers_v,0.045179,0.045179,6.095914e-02,5171
5,kiq026___ever_had_kidney_stones?,categorical,5.98,3,cramers_v,0.037789,0.037789,1.843674e-02,5593


In [65]:
numeric_features_ei = []
categorical_features_ei = []

for col in ei_focused_final_features:
    if infer_feature_type(X_train_ei[col]) == "numeric":
        numeric_features_ei.append(col)
    else:
        categorical_features_ei.append(col)

print("EI-focused numeric features:", numeric_features_ei)
print()
print("EI-focused categorical features:", categorical_features_ei)

EI-focused numeric features: ['med_count']

EI-focused categorical features: ['bpq020___ever_told_you_had_high_blood_pressure', 'kiq480___how_many_times_urinate_in_night?', 'paq650___vigorous_recreational_activities', 'mcq160a___ever_told_you_had_arthritis', 'kiq026___ever_had_kidney_stones?', 'kiq022___ever_told_you_had_weak/failing_kidneys?', 'dpq040___feeling_tired_or_having_little_energy', 'mcq160p___ever_told_you_had_copd_emphysema', 'kiq005___how_often_have_urinary_leakage?']


In [66]:
preprocessor_ei = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features_ei),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features_ei),
    ],
    remainder="drop"
)

In [67]:
logreg_pipeline_ei = Pipeline(steps=[
    ("preprocessor", preprocessor_ei),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

rf_pipeline_ei = Pipeline(steps=[
    ("preprocessor", preprocessor_ei),
    ("model", RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [68]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

logreg_cv_ei = cross_validate(logreg_pipeline_ei, X_train_ei, y_train_ei, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv_ei = cross_validate(rf_pipeline_ei, X_train_ei, y_train_ei, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary_ei = pd.DataFrame({
    "model": ["logreg_ei_focused", "random_forest_ei_focused"],
    "roc_auc_mean": [logreg_cv_ei["test_roc_auc"].mean(), rf_cv_ei["test_roc_auc"].mean()],
    "avg_precision_mean": [logreg_cv_ei["test_avg_precision"].mean(), rf_cv_ei["test_avg_precision"].mean()],
    "f1_mean": [logreg_cv_ei["test_f1"].mean(), rf_cv_ei["test_f1"].mean()],
    "precision_mean": [logreg_cv_ei["test_precision"].mean(), rf_cv_ei["test_precision"].mean()],
    "recall_mean": [logreg_cv_ei["test_recall"].mean(), rf_cv_ei["test_recall"].mean()],
}).sort_values("avg_precision_mean", ascending=False)

cv_summary_ei

,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
0,logreg_ei_focused,0.650822,0.142491,0.201111,0.123629,0.540136
1,random_forest_ei_focused,0.600243,0.112856,0.179765,0.129692,0.293782


In [69]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

logreg_cv_ei = cross_validate(logreg_pipeline_ei, X_train_ei, y_train_ei, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv_ei = cross_validate(rf_pipeline_ei, X_train_ei, y_train_ei, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary_ei = pd.DataFrame({
    "model": ["logreg_ei_focused", "random_forest_ei_focused"],
    "roc_auc_mean": [logreg_cv_ei["test_roc_auc"].mean(), rf_cv_ei["test_roc_auc"].mean()],
    "avg_precision_mean": [logreg_cv_ei["test_avg_precision"].mean(), rf_cv_ei["test_avg_precision"].mean()],
    "f1_mean": [logreg_cv_ei["test_f1"].mean(), rf_cv_ei["test_f1"].mean()],
    "precision_mean": [logreg_cv_ei["test_precision"].mean(), rf_cv_ei["test_precision"].mean()],
    "recall_mean": [logreg_cv_ei["test_recall"].mean(), rf_cv_ei["test_recall"].mean()],
}).sort_values("avg_precision_mean", ascending=False)

cv_summary_ei

,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
0,logreg_ei_focused,0.650822,0.142491,0.201111,0.123629,0.540136
1,random_forest_ei_focused,0.600243,0.112856,0.179765,0.129692,0.293782


In [74]:
best_pipeline_ei = logreg_pipeline_ei
best_model_name_ei = "logreg_ei_focused"

best_pipeline_ei.fit(X_train_ei, y_train_ei)

test_proba_ei = best_pipeline_ei.predict_proba(X_test_ei)[:, 1]
test_pred_ei = (test_proba_ei >= 0.50).astype(int)

print("Best EI-focused model:", best_model_name_ei)
print("ROC-AUC:", round(roc_auc_score(y_test_ei, test_proba_ei), 4))
print("Average Precision:", round(average_precision_score(y_test_ei, test_proba_ei), 4))
print("Precision:", round(precision_score(y_test_ei, test_pred_ei, zero_division=0), 4))
print("Recall:", round(recall_score(y_test_ei, test_pred_ei, zero_division=0), 4))
print("F1:", round(f1_score(y_test_ei, test_pred_ei, zero_division=0), 4))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_ei, test_pred_ei))
print()
print(classification_report(y_test_ei, test_pred_ei, zero_division=0))

Best EI-focused model: logreg_ei_focused
ROC-AUC: 0.6678
Average Precision: 0.1585
Precision: 0.1319
Recall: 0.5948
F1: 0.216

Confusion Matrix:
[[918 454]
 [ 47  69]]

              precision    recall  f1-score   support

           0       0.95      0.67      0.79      1372
           1       0.13      0.59      0.22       116

    accuracy                           0.66      1488
   macro avg       0.54      0.63      0.50      1488
weighted avg       0.89      0.66      0.74      1488



In [75]:
perm_ei = permutation_importance(
    best_pipeline_ei,
    X_test_ei,
    y_test_ei,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1
)

perm_df_ei = pd.DataFrame({
    "feature": X_test_ei.columns,
    "importance_mean": perm_ei.importances_mean,
    "importance_std": perm_ei.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df_ei

,feature,importance_mean,importance_std
0,bpq020___ever_told_you_had_high_blood_pressure,0.014632,0.007788
2,kiq480___how_many_times_urinate_in_night?,0.012532,0.009095
3,paq650___vigorous_recreational_activities,0.010053,0.004661
1,med_count,0.006487,0.011882
7,dpq040___feeling_tired_or_having_little_energy,0.005741,0.004018
5,kiq026___ever_had_kidney_stones?,0.005209,0.001649
4,mcq160a___ever_told_you_had_arthritis,0.002696,0.002346
6,kiq022___ever_told_you_had_weak/failing_kidneys?,0.002097,0.002190
9,kiq005___how_often_have_urinary_leakage?,0.001872,0.003032
8,mcq160p___ever_told_you_had_copd_emphysema,0.000776,0.000818


In [76]:
comparison_test_ei = pd.DataFrame([
    {
        "run": "set1_all_questionnaire",
        "model": best_model_name,
        "roc_auc": roc_auc_score(y_test, test_proba),
        "avg_precision": average_precision_score(y_test, test_proba),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "recall": recall_score(y_test, test_pred, zero_division=0),
        "f1": f1_score(y_test, test_pred, zero_division=0),
        "n_features": len(final_features),
    },
    {
        "run": "set2_ei_focused",
        "model": best_model_name_ei,
        "roc_auc": roc_auc_score(y_test_ei, test_proba_ei),
        "avg_precision": average_precision_score(y_test_ei, test_proba_ei),
        "precision": precision_score(y_test_ei, test_pred_ei, zero_division=0),
        "recall": recall_score(y_test_ei, test_pred_ei, zero_division=0),
        "f1": f1_score(y_test_ei, test_pred_ei, zero_division=0),
        "n_features": len(ei_focused_final_features),
    }
])

comparison_test_ei.sort_values("avg_precision", ascending=False)

,run,model,roc_auc,avg_precision,precision,recall,f1,n_features
1,set2_ei_focused,logreg_ei_focused,0.667814,0.158529,0.131931,0.594828,0.215962,10
0,set1_all_questionnaire,logreg,0.646338,0.144668,0.128364,0.534483,0.207012,30


# EI-FINAL-QUIZ-FEATURES (8)

In [77]:
# Final compact quiz run for electrolyte_imbalance
# Goal: test a small, product-friendly feature set

EI_FINAL_QUIZ_FEATURES = [
    "bpq020___ever_told_you_had_high_blood_pressure",
    "kiq480___how_many_times_urinate_in_night?",
    "paq650___vigorous_recreational_activities",
    "med_count",
    "dpq040___feeling_tired_or_having_little_energy",
    "kiq026___ever_had_kidney_stones?",
    "mcq160a___ever_told_you_had_arthritis",
    "kiq022___ever_told_you_had_weak/failing_kidneys?",
]

ei_final_quiz_available = [c for c in EI_FINAL_QUIZ_FEATURES if c in df.columns]
ei_final_quiz_missing = [c for c in EI_FINAL_QUIZ_FEATURES if c not in df.columns]

print("Available EI final quiz features:", len(ei_final_quiz_available))
print(ei_final_quiz_available)
print()
print("Missing requested features:", len(ei_final_quiz_missing))
print(ei_final_quiz_missing)

Available EI final quiz features: 8
['bpq020___ever_told_you_had_high_blood_pressure', 'kiq480___how_many_times_urinate_in_night?', 'paq650___vigorous_recreational_activities', 'med_count', 'dpq040___feeling_tired_or_having_little_energy', 'kiq026___ever_had_kidney_stones?', 'mcq160a___ever_told_you_had_arthritis', 'kiq022___ever_told_you_had_weak/failing_kidneys?']

Missing requested features: 0
[]


In [78]:
ei_final_quiz_audit_rows = []

for col in ei_final_quiz_available:
    s = df[col]
    ei_final_quiz_audit_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "n_unique": s.nunique(dropna=True),
        "example_values": str(get_example_values(s, n=10)),
        "is_binary_like": is_binary_series(s),
    })

ei_final_quiz_audit_df = pd.DataFrame(ei_final_quiz_audit_rows).sort_values(
    ["missing_pct", "n_unique", "feature"],
    ascending=[False, True, True]
)

ei_final_quiz_audit_df

,feature,dtype,missing_pct,n_unique,example_values,is_binary_like
1,kiq480___how_many_times_urinate_in_night?,float64,18.14,8,"[np.float64(0.0), np.float64(2.0), np.float64(...",False
4,dpq040___feeling_tired_or_having_little_energy,float64,13.03,6,"[np.float64(0.0), np.float64(2.0), np.float64(...",False
7,kiq022___ever_told_you_had_weak/failing_kidneys?,float64,6.20,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
5,kiq026___ever_had_kidney_stones?,float64,6.20,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
6,mcq160a___ever_told_you_had_arthritis,float64,6.20,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
2,paq650___vigorous_recreational_activities,float64,0.00,2,"[np.float64(1.0), np.float64(2.0)]",False
0,bpq020___ever_told_you_had_high_blood_pressure,float64,0.00,3,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
3,med_count,int64,0.00,21,"[np.int64(1), np.int64(3), np.int64(6), np.int...",False


In [79]:
ei_final_quiz_df = df[[TARGET] + ei_final_quiz_available].copy()

mask = ei_final_quiz_df[TARGET].notna()
X_eifq = ei_final_quiz_df.loc[mask, ei_final_quiz_available].copy()
y_eifq = ei_final_quiz_df.loc[mask, TARGET].astype(int).copy()

X_train_eifq, X_test_eifq, y_train_eifq, y_test_eifq = train_test_split(
    X_eifq, y_eifq,
    test_size=TEST_SIZE,
    stratify=y_eifq,
    random_state=RANDOM_STATE
)

print("EI final quiz train shape:", X_train_eifq.shape)
print("EI final quiz test shape:", X_test_eifq.shape)
print("Train target distribution:")
print(y_train_eifq.value_counts(normalize=True).round(4))

EI final quiz train shape: (5949, 8)
EI final quiz test shape: (1488, 8)
Train target distribution:
electrolyte_imbalance
0    0.9222
1    0.0778
Name: proportion, dtype: float64


In [81]:
eifq_univariate_rows = []

for col in ei_final_quiz_available:
    s = X_train_eifq[col]
    feature_type = infer_feature_type(s)

    row = {
        "feature": col,
        "feature_type": feature_type,
        "missing_pct_train": round(s.isna().mean() * 100, 2),
        "n_unique_train": s.nunique(dropna=True),
        "association_metric": None,
        "association_value": np.nan,
        "abs_association": np.nan,
        "p_value": np.nan,
        "n_train_non_null": int(s.notna().sum())
    }

    if feature_type == "categorical":
        valid = s.notna() & y_train_eifq.notna()
        if valid.sum() >= 30:
            try:
                cv = cramers_v(s[valid], y_train_eifq[valid])
                table = pd.crosstab(s[valid], y_train_eifq[valid])
                p_val = chi2_contingency(table)[1] if table.shape[0] > 1 and table.shape[1] > 1 else np.nan
                row["association_metric"] = "cramers_v"
                row["association_value"] = cv
                row["abs_association"] = abs(cv) if pd.notna(cv) else np.nan
                row["p_value"] = p_val
            except Exception:
                pass
    else:
        r, p = safe_pointbiserial(s, y_train_eifq)
        row["association_metric"] = "pointbiserial"
        row["association_value"] = r
        row["abs_association"] = abs(r) if pd.notna(r) else np.nan
        row["p_value"] = p

    eifq_univariate_rows.append(row)

eifq_univariate_df = pd.DataFrame(eifq_univariate_rows).sort_values(
    ["abs_association", "p_value"],
    ascending=[False, True]
)

eifq_univariate_df

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null
3,med_count,numeric,0.00,21,pointbiserial,0.129593,0.129593,1.061670e-23,5949
0,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.116671,0.116671,2.603849e-18,5949
1,kiq480___how_many_times_urinate_in_night?,categorical,17.92,8,cramers_v,0.095109,0.095109,1.980451e-07,4883
6,mcq160a___ever_told_you_had_arthritis,categorical,5.98,3,cramers_v,0.077041,0.077041,6.188193e-08,5593
2,paq650___vigorous_recreational_activities,categorical,0.00,2,cramers_v,0.064761,0.064761,5.883427e-07,5949
7,kiq022___ever_told_you_had_weak/failing_kidneys?,categorical,5.98,3,cramers_v,0.058777,0.058777,6.371589e-05,5593
4,dpq040___feeling_tired_or_having_little_energy,categorical,13.08,6,cramers_v,0.045179,0.045179,6.095914e-02,5171
5,kiq026___ever_had_kidney_stones?,categorical,5.98,3,cramers_v,0.037789,0.037789,1.843674e-02,5593


In [83]:
numeric_features_eifq = []
categorical_features_eifq = []

for col in ei_final_quiz_available:
    if infer_feature_type(X_train_eifq[col]) == "numeric":
        numeric_features_eifq.append(col)
    else:
        categorical_features_eifq.append(col)

print("EI final quiz numeric features:", numeric_features_eifq)
print()
print("EI final quiz categorical features:", categorical_features_eifq)

EI final quiz numeric features: ['med_count']

EI final quiz categorical features: ['bpq020___ever_told_you_had_high_blood_pressure', 'kiq480___how_many_times_urinate_in_night?', 'paq650___vigorous_recreational_activities', 'dpq040___feeling_tired_or_having_little_energy', 'kiq026___ever_had_kidney_stones?', 'mcq160a___ever_told_you_had_arthritis', 'kiq022___ever_told_you_had_weak/failing_kidneys?']


In [84]:
preprocessor_eifq = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features_eifq),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features_eifq),
    ],
    remainder="drop"
)

In [85]:
baseline_model_eifq = DummyClassifier(strategy="most_frequent")

baseline_model_eifq.fit(X_train_eifq, y_train_eifq)
baseline_pred_eifq = baseline_model_eifq.predict(X_test_eifq)

print("Baseline precision:", precision_score(y_test_eifq, baseline_pred_eifq, zero_division=0))
print("Baseline recall:", recall_score(y_test_eifq, baseline_pred_eifq, zero_division=0))
print("Baseline f1:", f1_score(y_test_eifq, baseline_pred_eifq, zero_division=0))

Baseline precision: 0.0
Baseline recall: 0.0
Baseline f1: 0.0


In [89]:
logreg_pipeline_eifq = Pipeline(steps=[
    ("preprocessor", preprocessor_eifq),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

rf_pipeline_eifq = Pipeline(steps=[
    ("preprocessor", preprocessor_eifq),
    ("model", RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [90]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

logreg_cv_eifq = cross_validate(logreg_pipeline_eifq, X_train_eifq, y_train_eifq, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv_eifq = cross_validate(rf_pipeline_eifq, X_train_eifq, y_train_eifq, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary_eifq = pd.DataFrame({
    "model": ["logreg_ei_final_quiz", "random_forest_ei_final_quiz"],
    "roc_auc_mean": [logreg_cv_eifq["test_roc_auc"].mean(), rf_cv_eifq["test_roc_auc"].mean()],
    "avg_precision_mean": [logreg_cv_eifq["test_avg_precision"].mean(), rf_cv_eifq["test_avg_precision"].mean()],
    "f1_mean": [logreg_cv_eifq["test_f1"].mean(), rf_cv_eifq["test_f1"].mean()],
    "precision_mean": [logreg_cv_eifq["test_precision"].mean(), rf_cv_eifq["test_precision"].mean()],
    "recall_mean": [logreg_cv_eifq["test_recall"].mean(), rf_cv_eifq["test_recall"].mean()],
}).sort_values("avg_precision_mean", ascending=False)

cv_summary_eifq

,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
0,logreg_ei_final_quiz,0.651655,0.147076,0.204880,0.125692,0.555213
1,random_forest_ei_final_quiz,0.604106,0.129870,0.184635,0.129471,0.321833


In [91]:
best_pipeline_eifq = logreg_pipeline_eifq
best_model_name_eifq = "logreg_ei_final_quiz"

best_pipeline_eifq.fit(X_train_eifq, y_train_eifq)

test_proba_eifq = best_pipeline_eifq.predict_proba(X_test_eifq)[:, 1]
test_pred_eifq = (test_proba_eifq >= 0.50).astype(int)

print("Best EI final quiz model:", best_model_name_eifq)
print("ROC-AUC:", round(roc_auc_score(y_test_eifq, test_proba_eifq), 4))
print("Average Precision:", round(average_precision_score(y_test_eifq, test_proba_eifq), 4))
print("Precision:", round(precision_score(y_test_eifq, test_pred_eifq, zero_division=0), 4))
print("Recall:", round(recall_score(y_test_eifq, test_pred_eifq, zero_division=0), 4))
print("F1:", round(f1_score(y_test_eifq, test_pred_eifq, zero_division=0), 4))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_eifq, test_pred_eifq))
print()
print(classification_report(y_test_eifq, test_pred_eifq, zero_division=0))

Best EI final quiz model: logreg_ei_final_quiz
ROC-AUC: 0.6721
Average Precision: 0.157
Precision: 0.1365
Recall: 0.6121
F1: 0.2233

Confusion Matrix:
[[923 449]
 [ 45  71]]

              precision    recall  f1-score   support

           0       0.95      0.67      0.79      1372
           1       0.14      0.61      0.22       116

    accuracy                           0.67      1488
   macro avg       0.55      0.64      0.51      1488
weighted avg       0.89      0.67      0.74      1488



In [92]:
perm_eifq = permutation_importance(
    best_pipeline_eifq,
    X_test_eifq,
    y_test_eifq,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1
)

perm_df_eifq = pd.DataFrame({
    "feature": X_test_eifq.columns,
    "importance_mean": perm_eifq.importances_mean,
    "importance_std": perm_eifq.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df_eifq

,feature,importance_mean,importance_std
0,bpq020___ever_told_you_had_high_blood_pressure,0.014057,0.007479
1,kiq480___how_many_times_urinate_in_night?,0.012126,0.008468
3,med_count,0.009671,0.010546
2,paq650___vigorous_recreational_activities,0.009434,0.004994
4,dpq040___feeling_tired_or_having_little_energy,0.005739,0.004184
5,kiq026___ever_had_kidney_stones?,0.004456,0.001557
6,mcq160a___ever_told_you_had_arthritis,0.002632,0.002432
7,kiq022___ever_told_you_had_weak/failing_kidneys?,0.000840,0.002325


In [93]:
combined_df_eifq = (
    eifq_univariate_df.merge(perm_df_eifq, on="feature", how="left")
    .merge(ei_final_quiz_audit_df[["feature", "missing_pct", "n_unique"]], on="feature", how="left")
)

combined_df_eifq = combined_df_eifq.sort_values(
    ["importance_mean", "abs_association"],
    ascending=[False, False]
)

combined_df_eifq

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null,importance_mean,importance_std,missing_pct,n_unique
1,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.116671,0.116671,2.603849e-18,5949,0.014057,0.007479,0.00,3
2,kiq480___how_many_times_urinate_in_night?,categorical,17.92,8,cramers_v,0.095109,0.095109,1.980451e-07,4883,0.012126,0.008468,18.14,8
0,med_count,numeric,0.00,21,pointbiserial,0.129593,0.129593,1.061670e-23,5949,0.009671,0.010546,0.00,21
4,paq650___vigorous_recreational_activities,categorical,0.00,2,cramers_v,0.064761,0.064761,5.883427e-07,5949,0.009434,0.004994,0.00,2
6,dpq040___feeling_tired_or_having_little_energy,categorical,13.08,6,cramers_v,0.045179,0.045179,6.095914e-02,5171,0.005739,0.004184,13.03,6
7,kiq026___ever_had_kidney_stones?,categorical,5.98,3,cramers_v,0.037789,0.037789,1.843674e-02,5593,0.004456,0.001557,6.20,3
3,mcq160a___ever_told_you_had_arthritis,categorical,5.98,3,cramers_v,0.077041,0.077041,6.188193e-08,5593,0.002632,0.002432,6.20,3
5,kiq022___ever_told_you_had_weak/failing_kidneys?,categorical,5.98,3,cramers_v,0.058777,0.058777,6.371589e-05,5593,0.000840,0.002325,6.20,3


In [94]:
comparison_test_eifq = pd.DataFrame([
    {
        "run": "set1_all_questionnaire",
        "model": best_model_name,
        "roc_auc": roc_auc_score(y_test, test_proba),
        "avg_precision": average_precision_score(y_test, test_proba),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "recall": recall_score(y_test, test_pred, zero_division=0),
        "f1": f1_score(y_test, test_pred, zero_division=0),
        "n_features": len(final_features),
    },
    {
        "run": "set2_ei_focused",
        "model": best_model_name_ei,
        "roc_auc": roc_auc_score(y_test_ei, test_proba_ei),
        "avg_precision": average_precision_score(y_test_ei, test_proba_ei),
        "precision": precision_score(y_test_ei, test_pred_ei, zero_division=0),
        "recall": recall_score(y_test_ei, test_pred_ei, zero_division=0),
        "f1": f1_score(y_test_ei, test_pred_ei, zero_division=0),
        "n_features": len(ei_focused_final_features),
    },
    {
        "run": "set3_ei_final_quiz_8_features",
        "model": best_model_name_eifq,
        "roc_auc": roc_auc_score(y_test_eifq, test_proba_eifq),
        "avg_precision": average_precision_score(y_test_eifq, test_proba_eifq),
        "precision": precision_score(y_test_eifq, test_pred_eifq, zero_division=0),
        "recall": recall_score(y_test_eifq, test_pred_eifq, zero_division=0),
        "f1": f1_score(y_test_eifq, test_pred_eifq, zero_division=0),
        "n_features": len(ei_final_quiz_available),
    }
])

comparison_test_eifq.sort_values("avg_precision", ascending=False)

,run,model,roc_auc,avg_precision,precision,recall,f1,n_features
1,set2_ei_focused,logreg_ei_focused,0.667814,0.158529,0.131931,0.594828,0.215962,10
2,set3_ei_final_quiz_8_features,logreg_ei_final_quiz,0.672059,0.157008,0.136538,0.612069,0.223270,8
0,set1_all_questionnaire,logreg,0.646338,0.144668,0.128364,0.534483,0.207012,30


In [95]:
threshold_rows_eifq = []

for t in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    pred_t = (test_proba_eifq >= t).astype(int)
    threshold_rows_eifq.append({
        "threshold": t,
        "precision": precision_score(y_test_eifq, pred_t, zero_division=0),
        "recall": recall_score(y_test_eifq, pred_t, zero_division=0),
        "f1": f1_score(y_test_eifq, pred_t, zero_division=0),
        "positives_predicted": int(pred_t.sum())
    })

threshold_df_eifq = pd.DataFrame(threshold_rows_eifq).sort_values("f1", ascending=False)
threshold_df_eifq

,threshold,precision,recall,f1,positives_predicted
6,0.50,0.136538,0.612069,0.223270,520
5,0.40,0.099165,0.818966,0.176909,958
4,0.35,0.085499,0.879310,0.155844,1193
3,0.30,0.081001,0.948276,0.149254,1358
2,0.25,0.078178,0.991379,0.144928,1471
1,0.20,0.078062,1.000000,0.144819,1486
0,0.15,0.077957,1.000000,0.144638,1488


# Save the winner model via joblib

In [96]:
X_eifq_full = df.loc[df[TARGET].notna(), EI_FINAL_QUIZ_FEATURES].copy()
y_eifq_full = df.loc[df[TARGET].notna(), TARGET].astype(int).copy()

final_pipeline_eifq = Pipeline(steps=[
    ("preprocessor", preprocessor_eifq),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

final_pipeline_eifq.fit(X_eifq_full, y_eifq_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [97]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "electrolyte_imbalance_logreg_final_quiz.joblib"
joblib.dump(final_pipeline_eifq, model_path)

print("Saved model to:", model_path.resolve())

Saved model to: C:\Users\Philipp\AIBootcamp\halfFull\models\electrolyte_imbalance_logreg_final_quiz.joblib


In [99]:
import json

model_metadata = {
    "model_name": "electrolyte_imbalance_logreg_final_quiz",
    "target": "electrolyte_imbalance",
    "features": EI_FINAL_QUIZ_FEATURES,
    "threshold": 0.50,
    "metrics": {
        "roc_auc": float(0.672059),
        "avg_precision": float(0.157008),
        "precision": float(0.136538),
        "recall": float(0.612069),
        "f1": float(0.223270),
    }
}

metadata_path = MODEL_DIR / "electrolyte_imbalance_logreg_final_quiz_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=2)

print("Saved metadata to:", metadata_path.resolve())

Saved metadata to: C:\Users\Philipp\AIBootcamp\halfFull\models\electrolyte_imbalance_logreg_final_quiz_metadata.json
